# Unix 1: finding your location and navigating the filesystem

This practical connects the graphical file browser to the Unix command line on **macOS**, **Ubuntu**, and **Windows Subsystem for Linux (WSL)**. Work through it in order and use the **predict–run–explain** cycle: predict what a cell will report, run it, then explain the result to a partner.

> All practical file creation happens in a temporary practice folder. The course files are not changed.

## Learning outcomes

By the end of this notebook you should be able to:

1. distinguish the operating system, shell, home directory, filesystem root, and current working directory;
2. find where a Python/Jupyter process is currently working without relying on a particular username;
3. navigate with `pwd`, `ls`, and `cd`, using absolute and relative paths;
4. show the same folder in Finder, Ubuntu Files, or Windows File Explorer;
5. translate common Windows and WSL paths; and
6. recover when you are lost or a path containing spaces does not work.

## First: four different questions

These questions sound similar, but their answers can differ:

| Question | What answers it? | Example |
|---|---|---|
| Which operating environment am I using? | Python's platform information | macOS, Ubuntu, or Ubuntu inside WSL |
| Which user am I? | `whoami` or Python | `student` |
| Where is my home directory? | `echo "$HOME"` or `Path.home()` | `/home/student` |
| Where is this process working now? | `pwd` or `Path.cwd()` | `/home/student/course` |

**Important:** Python can reliably report the notebook kernel's **current working directory**. It cannot always discover the saved location of the `.ipynb` file, because a Jupyter server can open a notebook while starting its kernel somewhere else.

In [ ]:
from pathlib import Path
import os
import platform

def running_in_wsl():
    """Return True when this Python process appears to be inside WSL."""
    if os.environ.get("WSL_DISTRO_NAME"):
        return True
    try:
        return "microsoft" in Path("/proc/version").read_text().lower()
    except OSError:
        return False

system = platform.system()
environment = (
    f"WSL ({os.environ.get('WSL_DISTRO_NAME', 'Linux distribution')})"
    if running_in_wsl()
    else {"Darwin": "macOS", "Linux": "Linux/Ubuntu"}.get(system, system)
)

print(f"Environment:               {environment}")
print(f"Python platform:           {platform.platform()}")
print(f"User:                      {os.environ.get('USER', os.environ.get('USERNAME', 'unknown'))}")
print(f"Home directory:            {Path.home()}")
print(f"Kernel working directory:  {Path.cwd()}")

### Stop and interpret the report

Write down the final two paths. Are they the same? They do not have to be.

On WSL, `Path.home()` normally reports the **Linux** home directory, such as `/home/student`, not the Windows profile directory such as `C:\Users\Student`. That is expected: Python is running inside the Linux environment.

## The five path symbols you need first

| Symbol | Meaning | Typical command |
|---|---|---|
| `/` | filesystem root: the top of the Unix tree | `cd /` |
| `~` | your home directory | `cd ~` |
| `.` | the current directory | `ls .` |
| `..` | the parent of the current directory | `cd ..` |
| `-` | the previous working directory (for `cd`) | `cd -` |

Do not confuse `/` with `~`. The root belongs to the whole filesystem; home is your personal starting area within it.

In [ ]:
current = Path.cwd().resolve()
print("Current location and its ancestors:")
for level, location in enumerate((current, *current.parents)):
    label = "current directory" if level == 0 else f"parent {level}"
    print(f"{label:18} {location}")

## Absolute and relative paths

An **absolute path** starts at `/` and identifies a location independently of where you are now: `/home/student/course/data`.

A **relative path** starts from the current working directory: `data`, `../data`, or `./data`. Relative paths are convenient, but their meaning changes when the current working directory changes.

Read a Unix path from left to right. Each `/` means “inside”. For example, `/home/student/course/data` means: root → `home` → `student` → `course` → `data`.

## Build a safe practice filesystem

The next cell makes a uniquely named temporary directory and a small folder tree. It also places its location in an environment variable called `NAV_PRACTICE`, so Bash cells can use it without knowing your username or operating system.

In [ ]:
import tempfile

practice = Path(tempfile.mkdtemp(prefix="unix1_navigation_")).resolve()
for folder in [
    practice / "Projects" / "Field work" / "data",
    practice / "Projects" / "Lab_work" / "results",
    practice / "Notes",
]:
    folder.mkdir(parents=True, exist_ok=True)

(practice / "Projects" / "Field work" / "data" / "samples.csv").write_text(
    "sample,value\nA,12\nB,15\n"
)
(practice / "Notes" / "where_am_i.txt").write_text("Navigation practice\n")
os.environ["NAV_PRACTICE"] = str(practice)

print(f"Practice folder: {practice}")
print("This folder is outside the course materials and can be deleted later.")

## Navigate at the command line

The core loop is:

1. `pwd` — print where I am;
2. `ls` — list what is here;
3. `cd PATH` — change to another directory.

Before running the next cell, predict the path printed by each `pwd`. Notice that changing directory inside this Bash cell does **not** change the Python kernel's working directory: the Bash process ends when the cell finishes.

In [ ]:
%%bash
cd "$NAV_PRACTICE"
echo "1. Start"
pwd
ls

echo
echo "2. Move using a relative path"
cd Projects/Lab_work
pwd
ls

echo
echo "3. Move to the parent"
cd ..
pwd
ls

## Paths containing spaces

A space normally separates command arguments. Protect a path containing spaces by quoting the entire path: `cd "Field work"`. Tab completion can also add the necessary escaping for you.

Run the next cell, then remove the quotation marks and run it again. Read the error before restoring the working version.

In [ ]:
%%bash
cd "$NAV_PRACTICE/Projects/Field work/data"
pwd
ls -l
cat samples.csv

## Open the command-line location in the graphical file browser

Run the appropriate command **in a terminal**, from the folder you want to display:

| Environment | Command | Graphical application |
|---|---|---|
| macOS | `open .` | Finder |
| Ubuntu desktop | `xdg-open .` | Default file manager, commonly Files |
| WSL | `explorer.exe .` | Windows File Explorer |

The dot matters: it means “open the current directory”. This notebook will recommend a command but will not launch an application automatically.

In [ ]:
if running_in_wsl():
    gui_command = "explorer.exe ."
elif system == "Darwin":
    gui_command = "open ."
elif system == "Linux":
    gui_command = "xdg-open ."
else:
    gui_command = "Use your system's file browser"

print(f"Suggested terminal command for this environment: {gui_command}")
print(f"To practise, first run: cd {practice!s}")

## Move from the GUI back to the terminal

The goal is to obtain the folder's path, then pass that path to `cd`.

### macOS Finder

Choose **View → Show Path Bar**. Control-click a folder in the path bar and choose **Copy … as Pathname**. In the terminal, type `cd ` and paste the path. Finder can also open a typed path through **Go → Go to Folder…**.

### Ubuntu Files (GNOME)

Press **Ctrl+L** to turn the location bar into editable path text. Copy it, type `cd ` in the terminal, paste, and press Enter. You can also paste a Unix path into this bar and press Enter to visit it.

### Windows File Explorer with WSL

From a WSL terminal, `explorer.exe .` is the simplest bridge. Linux files can also appear in Explorer under a path such as `\\wsl.localhost\Ubuntu\home\student\course`; the distribution name may not be `Ubuntu`.

## WSL has two kinds of location

| Seen from WSL | Seen from Windows | Meaning |
|---|---|---|
| `/home/student/project` | `\\wsl.localhost\Ubuntu\home\student\project` | A project stored in the Linux filesystem |
| `/mnt/c/Users/Student/Documents` | `C:\Users\Student\Documents` | Windows C: drive mounted inside WSL |

Use `wslpath` when you need a reliable conversion rather than rewriting a path by hand:

```bash
wslpath -w /home/student/project
wslpath 'C:\Users\Student\Documents'
```

For Linux command-line projects, Microsoft recommends keeping working files in the Linux filesystem (for example under `~/projects`) for better performance. Use `/mnt/c/...` when you deliberately need files stored on the Windows side.

## What does the Jupyter file browser show?

JupyterLab's file browser shows locations available below the server's configured starting directory. Its top level is not necessarily Unix `/`, and it may prevent navigation above that starting point. Meanwhile, code runs relative to the kernel's current working directory.

Therefore, when a data file is reported as missing:

1. run `Path.cwd()` in Python or `pwd` in a shell cell;
2. check the exact spelling and capitalisation with `Path.cwd().iterdir()` or `ls`;
3. compare the path with the location shown in the Jupyter file browser; and
4. use an explicit relative or absolute path instead of guessing.

In [ ]:
print(f"Python is still working in: {Path.cwd()}")
print("The Bash cell's cd did not change this.")
print("\nItems in the practice directory, inspected with Python:")
for item in sorted(practice.iterdir()):
    kind = "directory" if item.is_dir() else "file"
    print(f"- {item.name} ({kind})")

## Guided challenge: GUI and command line describe the same tree

Open a separate terminal and complete these tasks. Do not paste commands until you have predicted their effect.

1. Copy the printed practice-folder path from above. Run `cd "PASTE_THE_PATH_HERE"`.
2. Confirm the location with `pwd`, then list it with `ls`.
3. Enter `Projects`, then `Field work`, then `data`, using relative paths.
4. Display `samples.csv` with `cat`.
5. Return to `Projects` using `..`.
6. Jump to your home directory with `cd ~`, then return with `cd -`.
7. Open the current folder in the appropriate graphical file browser.
8. In the GUI, open `Lab_work/results`. Copy or reveal its path and use that path with `cd` in the terminal.

**Evidence to show a partner:** the output of `pwd`, the GUI breadcrumb/address, and one sentence explaining why they identify the same folder even if their formatting differs.

## If you are lost: a recovery checklist

```bash
pwd                 # Where am I?
ls                  # What is here?
ls ..               # What is one level above?
cd ~                # Return to my home directory
cd -                # Return to my previous directory
```

Then check:

- Did you type the same upper- and lower-case letters? Unix filenames are normally case-sensitive.
- Does the path contain a space? Quote it.
- Did you copy a Windows path into WSL? Convert it with `wslpath`.
- Are you in a remote SSH session? A remote machine has a different filesystem from your laptop.
- Are you looking at the Jupyter server root rather than Unix root? Use `pwd` to establish the kernel's location.

## Check your understanding

Answer before revealing the explanations below.

1. If `pwd` reports `/home/lee/course` and you run `cd ..`, where will you be?
2. Why can `Path.cwd()` differ from the folder containing the notebook file?
3. What does `/mnt/c/Users/Lee` tell you about the environment and storage location?
4. Why does `cd Field work` fail while `cd "Field work"` works?
5. Which command opens the current WSL directory in Windows File Explorer?

<details>
<summary>Reveal explanations</summary>

1. `/home/lee`, because `..` means the parent directory.
2. Jupyter can start the kernel in a configured working directory independently of where the notebook is saved.
3. You are viewing the Windows C: drive through its usual WSL mount point.
4. The shell treats an unquoted space as an argument separator.
5. `explorer.exe .`

</details>

## Optional cleanup

The practice folder is in your system's temporary-file area. If you have finished and want to remove it, run the next cell. It checks that the folder name begins with the expected prefix before deleting it. If you want to repeat the exercises, leave it in place or rerun the setup cell to create a fresh tree.

In [ ]:
# Optional: remove only the temporary practice directory made by this notebook.
import shutil

if practice.is_dir() and practice.name.startswith("unix1_navigation_"):
    shutil.rmtree(practice)
    print(f"Removed {practice}")
else:
    print("Safety check failed; nothing was removed.")

## Platform references

- [Microsoft Learn: WSL interop and filesystem paths](https://learn.microsoft.com/en-us/windows/dev-environment/wsl-interop)
- [Microsoft Learn: working across Windows and WSL filesystems](https://learn.microsoft.com/en-us/windows/wsl/filesystems)
- [Apple Support: show and copy a folder path in Finder](https://support.apple.com/en-gb/guide/mac-help/mchlp1774/mac)
- [GNOME Help: show or enter a directory path in Files](https://help.gnome.org/gnome-help/files-show-directory-path.html)